In [21]:
import torch
from dataExtraction import get_proteomics, get_genes
from utils import run_experiment
from models import LinearModel, OneHiddenLayer, ActuallyDeepNetwork
from losses import VariancePenaltyLoss
import torch.nn as nn
import pandas as pd
import numpy as np

# Checking for Metal availabilty
if torch.mps.is_available():
    device = torch.device("mps")
    print("M4 GPU acceleration with Metal is ready")
else:
    device = torch.device("cpu")
    print("Metal not found, back on cpu")

M4 GPU acceleration with Metal is ready


In [22]:
CONFIG = {
    "n_top_genes"  : 300,
    "n_splits"     : 5,
    "batch_size"   : 128,
    "lr"           : 1e-3,
    "weight_decay" : 0.01,
    "n_neighbors"  : 5,
    "max_epochs"   : 500,
    "patience"     : 20,
    "min_delta"    : 1e-4,
    "device"       : device,
}


In [23]:
# Data Loading
X_prot, X_meta       = get_proteomics("data/sound_life_all_olink.csv", only_day_0=False)
Y, gene_names, ids   = get_genes("data/sound-life_whole-blood_no-stim.h5ad", X_prot, n_top_genes=CONFIG["n_top_genes"])

proteomics_raw    = X_prot.loc[ids].values
metadata          = X_meta.loc[ids]
sex_encoded       = pd.get_dummies(metadata['subject.biologicalSex'], drop_first=True)
metadata_numeric  = pd.concat([sex_encoded, metadata['sample.subjectAgeAtDraw']], axis=1).astype(float).values
groups            = pd.factorize(metadata['subject.subjectGuid'])[0]

/Users/paulmaricelle/Stanford/mini-projet/dataExtraction.py:7: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


There are : 32 samples in X without corresponding value in Y


In [28]:
# Experiments : They require one model and one loss
from functools import partial

experiments = {
    "Linear + MSE"          : (LinearModel,    nn.MSELoss()),
    "Linear + VarPenalty"   : (LinearModel,    VariancePenaltyLoss(alpha=0.5)),
    "Linear + HubertLoss"   : (LinearModel, nn.HuberLoss()),
    "OneHidden + MSE"       : (partial(OneHiddenLayer, hidden=4096), nn.MSELoss()),
    "OneHidden + VarPenalty": (partial(OneHiddenLayer, hidden=4096), VariancePenaltyLoss(alpha=0.5)),
    "LargerNetwork + MSE"   : (ActuallyDeepNetwork, nn.MSELoss())
}

In [29]:
# Computing results and displaying them as a spreadsheet (the folds are the same for each experiment as GroupKFold has shuffle=False by default)
results = {}
for name, (model_class, loss_fn) in experiments.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    results[name] = run_experiment(model_class, loss_fn, proteomics_raw, Y, metadata_numeric, groups, CONFIG)

print(f"\n{'Experiment':<30} | {'Median r':>10} | {'Ratio std':>10}")
print("-" * 55)
for name, scores in results.items():
    median_r  = np.mean(scores["median_r"])
    std_r     = np.std(scores["median_r"])
    ratio_std = np.mean(scores["ratio_std"])
    print(f"{name:<30} | {median_r:.3f} ± {std_r:.3f} | {ratio_std:.3f}")


Linear + MSE

--- Fold 1/5 ---
Epoch    1 | Train loss : 0.8745 | Val loss : 0.7269 | No improve : 0/20
Epoch   10 | Train loss : 0.2391 | Val loss : 0.5600 | No improve : 0/20
Epoch   20 | Train loss : 0.2110 | Val loss : 0.5430 | No improve : 3/20
Epoch   30 | Train loss : 0.2165 | Val loss : 0.5373 | No improve : 4/20
Epoch   40 | Train loss : 0.2150 | Val loss : 0.5413 | No improve : 14/20
Epoch   50 | Train loss : 0.2130 | Val loss : 0.5330 | No improve : 3/20
Epoch   60 | Train loss : 0.2142 | Val loss : 0.5286 | No improve : 1/20
Epoch   70 | Train loss : 0.2167 | Val loss : 0.5446 | No improve : 9/20
Epoch   80 | Train loss : 0.2079 | Val loss : 0.5445 | No improve : 19/20
Early stopping at epoch 80 — best val loss : 0.5286

--- Fold 2/5 ---
Epoch    1 | Train loss : 0.8743 | Val loss : 0.7268 | No improve : 0/20
Epoch   10 | Train loss : 0.2394 | Val loss : 0.5611 | No improve : 0/20
Epoch   20 | Train loss : 0.2035 | Val loss : 0.5526 | No improve : 1/20
Epoch   30 | Train l